# TinyStories Phase 2 Scale-Up: Large PARF / FockPARF / Hybrid with SQ3 V_θ

**v1 — Phase 2 large-scale experiments**

## What is new vs. the v3 notebook (`tinystories_parf_vs_fockparf.ipynb`)

| Aspect | v3 (S1–S4) | Phase-2 large (L1–L3) |
|---|---|---|
| Embedding dim `d` | 256 | **512** |
| PARF depth `L` | 8 (pure) / 4 (hybrid) | **12 (pure) / 6 (hybrid)** |
| Attention layers | 0 / 4 | 0 / **8** |
| Attractor components `K` | 4 | **8** |
| Training steps | 16 000 | **32 000** |
| `_layer_step` V_θ gradient | hot-swap only (Phase 1) | **`analytical_grad` in-loop (Phase 2)** |
| `create_graph` scope | V_θ + V_φ (~2.5 M params) | **V_φ only (~660 K params, 3.8× smaller)** |

Phase 2 wires `MixtureQuadraticVTheta.analytical_grad()` directly into `_layer_step`,
replacing the `autograd.grad(U_total, h)` call with a cheap analytical matvec for V_θ
plus a much lighter `autograd.grad(U_phi, h)`.  At `d=512` the `create_graph` graph
is ~3.8× smaller, making 32 k steps at `d=512` feasible on a Colab A100.

## Experiment cells

| Cell | Architecture | `d` | `L` | `K` | Steps | Goal |
|---|---|---|---|---|---|---|
| **L1** | Pure PARF (SparsePARFLM) | 512 | 12 | 8 | 32 000 | Best pure-PARF PPL at scale |
| **L2** | Pure FockPARF | 512 | 10 | 8 | 32 000 | Fock mechanism at d=512 |
| **L3** | Hybrid FockPARF+Attn | 512 | 6+8 | 8 | 32 000 | Close gap to attention baseline |

**Recommended run order: L3 → L1 → L2** (L3 has the best expected PPL and highest research priority).

Reference baselines (TinyStories):
- Attention GPT-2-style 8L d=256: **~7.81 PPL**
- PARF S1 d=256 16k steps: **~26 PPL**
- PARF v3 S1 SQ3 K=4 d=256 16k: **~TBD**

## 0. Environment setup + cell selector

In [ ]:
CELL = 'L3'       # one of: 'L1' | 'L2' | 'L3'  -- run L3 first!
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_tinystories_phase2_large'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf' / 'results' / 'tinystories_phase2_large'

CONSERV_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'   # contains data_module.py
PARF_DIR    = CONSERV_DIR / 'parf'
SARF_DIR    = CONSERV_DIR / 'sarf_mass_variant'
HYBRID_DIR  = CONSERV_DIR / 'hybrid'
DATA_DIR    = CONSERV_DIR / 'data'

for _d in (CONSERV_DIR, PARF_DIR, SARF_DIR, HYBRID_DIR, DATA_DIR):
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))
for _d in (PARF_DIR / 'scripts',
           CONSERV_DIR / 'energetic_minima'):
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))

RESULTS_ROOT = GDRIVE_OUT / CELL / f'seed{SEED}'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT   = {REPO_ROOT}')
print(f'GDRIVE_OUT  = {GDRIVE_OUT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')

## 1. GPU check

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled (SPLM autograd.grad sensitivity)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS (Apple Silicon)')
else:
    device = 'cpu'
    print('WARNING: no GPU detected; training will be very slow')

print(f'device = {device}')
rng = np.random.default_rng(SEED)

## 2. Experiment recipes (Phase 2 Large)

All three cells use `MixtureQuadraticVTheta(K=8)` (SQ3) with Phase 2:
the analytical gradient is dispatched automatically inside `_layer_step`
— no extra code is needed beyond the hot-swap in Cell 4.

**V_φ hyperparameters are scaled up** (`V_PHI_D_TYPE=64`, `V_PHI_D_ANGLE=32`)
to match the larger embedding dimension `d=512` and give V_φ more
representational capacity for modelling token-pair interactions.

In [ ]:
SCALE_RECIPES = {
    # ── L1: Pure PARF at d=512 ───────────────────────────────────────────────
    # Tests whether the Phase-2 analytical gradient at d=512 (L=12, K=8)
    # meaningfully reduces PPL compared to v3 S1 (d=256, L=8, K=4).
    # Expected: ~4-6 PPL improvement over v3 S1 from scale alone.
    'L1': {
        'desc': 'Phase-2 PARF d=512 L=12 K=8 (32k steps)',
        'model': 'parf',
        'd': 512, 'L': 12,
        'v_theta_kind': 'sq3', 'v_theta_K': 8,
        'v_hidden': 2048, 'v_depth': 3,      # unused (SQ3 active)
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 0, 'creation_gate_hidden': 0,
        'top_k': 8, 'score_head_hidden': 64,
        'lambda_v': 1e-2, 'steps': 32000,
        'batch': 8, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    # ── L2: Pure FockPARF at d=512 ───────────────────────────────────────────
    # Tests whether the Fock creation/destruction mechanism at d=512
    # can close the gap vs pure PARF.  n_registers=64 (vs 32 in v3 S2)
    # to give the register bank more capacity at the wider embedding.
    'L2': {
        'desc': 'Phase-2 FockPARF d=512 L=10 K=8 M=64 (32k steps)',
        'model': 'fock_parf',
        'd': 512, 'L': 10,
        'v_theta_kind': 'sq3', 'v_theta_K': 8,
        'v_hidden': 2048, 'v_depth': 3,
        'n_attn': 0, 'n_head': 0, 'mlp_mult': 0,
        'n_registers': 64, 'creation_gate_hidden': 128,
        'top_k': 8, 'score_head_hidden': 64,
        'lambda_v': 1e-2, 'steps': 32000,
        'batch': 8, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
    # ── L3: Hybrid FockPARF+Attn at d=512 ───────────────────────────────────
    # Best-effort attempt to close the PPL gap vs the attention baseline.
    # 8 attention layers handle long-range context; 6 PARF layers enforce
    # conservative dynamics.  Total depth = 14 layers, ~GPT-2-medium scale.
    # The attention layers use n_head=8, mlp_mult=4 (standard GPT-2 ratios).
    'L3': {
        'desc': 'Phase-2 Hybrid FockPARF+Attn d=512 (6P+8A) K=8 (32k steps)',
        'model': 'hybrid_fock_parf',
        'd': 512, 'L': 6,
        'v_theta_kind': 'sq3', 'v_theta_K': 8,
        'v_hidden': 2048, 'v_depth': 3,
        'n_attn': 8, 'n_head': 8, 'mlp_mult': 4,
        'n_registers': 64, 'creation_gate_hidden': 128,
        'top_k': 8, 'score_head_hidden': 64,
        'lambda_v': 1e-2, 'steps': 32000,
        'batch': 4, 'block': 512,
        'init_gamma': 0.10, 'fixed_gamma': None,
    },
}
if CELL not in SCALE_RECIPES:
    raise ValueError(f'CELL must be one of {list(SCALE_RECIPES)}; got {CELL!r}')

recipe = SCALE_RECIPES[CELL]
LAMBDA_V       = recipe['lambda_v']
STEPS          = recipe['steps']
D              = recipe['d']
L              = recipe['L']
V_THETA_KIND   = recipe.get('v_theta_kind', 'mlp')
V_THETA_K      = recipe.get('v_theta_K', 8)
V_HIDDEN       = recipe['v_hidden']
V_DEPTH        = recipe['v_depth']
N_ATTN         = recipe['n_attn']
N_HEAD         = recipe['n_head']
MLP_MULT       = recipe['mlp_mult']
N_REGISTERS    = recipe['n_registers']
CREATION_GATE_HIDDEN = recipe['creation_gate_hidden']
TOP_K          = recipe['top_k']
SCORE_HEAD_HIDDEN = recipe['score_head_hidden']
BATCH          = recipe['batch']
BLOCK          = recipe['block']
INIT_GAMMA     = recipe.get('init_gamma', 0.10)
FIXED_GAMMA    = recipe.get('fixed_gamma', None)
MODEL_KIND     = recipe['model']

VOCAB_SIZE  = 50257
MAX_LEN     = 1024
DT          = 1.0
V_PHI_KIND  = 'structural'
# Scaled-up V_phi hidden dims to match d=512
V_PHI_D_TYPE      = 64
V_PHI_D_ANGLE     = 32
V_PHI_PHI_HIDDEN  = 32
V_PHI_THETA_HIDDEN = 32
V_PHI_MLP_HIDDEN  = 64
GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN  = 0.1
LR = 3e-4        # slightly lower LR for larger model stability
WD = 0.01
WARMUP = 1200    # longer warmup for larger model
GRAD_CLIP = 1.0
EVAL_INTERVAL = 500
EVAL_ITERS = 40
LOG_INTERVAL = 100
STACK_DISCIPLINE = True

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  model={MODEL_KIND}  d={D}  L={L}  V_theta={V_THETA_KIND}(K={V_THETA_K})  M={N_REGISTERS}')
print(f'  n_attn={N_ATTN} n_head={N_HEAD}  top_k={TOP_K}  lambda_V={LAMBDA_V}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}')
print(f'  init_gamma={INIT_GAMMA}  fixed_gamma={FIXED_GAMMA}')
print(f'  V_phi: d_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')

## 3. Load TinyStories

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

## 4. Build model (Phase 2 hot-swap)

After model instantiation the MLP `V_theta` is replaced with
`MixtureQuadraticVTheta(d, K)`.  Because `model_parf.py` and
`model_parf_sparse.py` now check `_has_analytical_grad(self.V_theta)`
inside `_layer_step`, the Phase-2 analytical-gradient path activates
**automatically** — no further code changes are required.

In [ ]:
from parf.model_fock_parf import FockPARFLM, FockPARFConfig
from parf.model_structured_vtheta import (
    MixtureQuadraticVTheta, QuadraticWellVTheta,
    LowRankQuadraticVTheta, HybridQuadraticVTheta,
)
from parf.model_hybrid_fock_parf import HybridFockPARF, HybridFockPARFConfig
from parf.model_parf_sparse import SparsePARFLM, SparsePARFConfig
from parf.model_parf import _has_analytical_grad
from sarf_mass_variant.model_sarf_mass import causal_cumulative_mean
import torch.nn.functional as F_torch

SCALEUP_LOGFREQ = CONSERV_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ   = GDRIVE_OUT / 'logfreq_surprisal_tinystories.npy'

if SCALEUP_LOGFREQ.exists():
    LOGFREQ_PATH = SCALEUP_LOGFREQ
    print(f'Using bundled logfreq: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

torch.manual_seed(SEED)

gamma_kwargs = dict(init_gamma=INIT_GAMMA)
if FIXED_GAMMA is not None:
    gamma_kwargs['fixed_gamma'] = FIXED_GAMMA

parf_base_kw = dict(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    v_phi_kind=V_PHI_KIND,
    v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
    v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
    v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
    v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD_HIDDEN,
    gumbel_tau_init=GUMBEL_TAU_INIT,
    gumbel_tau_min=GUMBEL_TAU_MIN,
    **gamma_kwargs,
)

if MODEL_KIND == 'hybrid_fock_parf':
    cfg = HybridFockPARFConfig(
        **parf_base_kw,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
        n_attn=N_ATTN, n_head=N_HEAD, mlp_mult=MLP_MULT,
    )
    model = HybridFockPARF(cfg).to(device)
    print(f'Hybrid FockPARF: n_attn={N_ATTN}, L={L} FockPARF layers')
elif MODEL_KIND == 'parf':
    cfg = SparsePARFConfig(**parf_base_kw)
    model = SparsePARFLM(cfg).to(device)
    print(f'SparsePARFLM: L={L}')
else:  # fock_parf
    cfg = FockPARFConfig(
        **parf_base_kw,
        n_registers=N_REGISTERS,
        creation_gate_hidden=CREATION_GATE_HIDDEN,
        stack_discipline=STACK_DISCIPLINE,
    )
    model = FockPARFLM(cfg).to(device)
    print(f'FockPARF: L={L}, M={N_REGISTERS}')

n_total   = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi   = sum(p.numel() for p in model.V_phi.parameters()) if hasattr(model, 'V_phi') else 0
print(f'params (before swap): total={n_total:,}  V_theta={n_v_theta:,}  V_phi={n_v_phi:,}')

# ── Phase-2 V_theta hot-swap ─────────────────────────────────────────────────
# After swap, _layer_step sees analytical_grad and routes through Phase-2 path.
if V_THETA_KIND == 'sq3':
    _old_vt_params = n_v_theta
    model.V_theta = MixtureQuadraticVTheta(d=D, K=V_THETA_K).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    n_total   = sum(p.numel() for p in model.parameters())
    assert _has_analytical_grad(model.V_theta), 'Phase-2 dispatch check failed'
    print(f'[SQ3 Phase-2] V_theta swapped: MixtureQuadraticVTheta(K={V_THETA_K})')
    print(f'  V_theta params: {_old_vt_params:,} → {n_v_theta:,}')
    print(f'  autograd.grad scope: V_theta+V_phi ({_old_vt_params+n_v_phi:,}) → '
          f'V_phi only ({n_v_phi:,})  '
          f'({100*n_v_phi/(_old_vt_params+n_v_phi):.0f}% of original)')
elif V_THETA_KIND == 'sq1':
    model.V_theta = QuadraticWellVTheta(d=D).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    n_total   = sum(p.numel() for p in model.parameters())
    assert _has_analytical_grad(model.V_theta)
    print(f'[SQ1 Phase-2] V_theta swapped: QuadraticWellVTheta, params={n_v_theta:,}')
elif V_THETA_KIND == 'sq2':
    model.V_theta = LowRankQuadraticVTheta(d=D, rank=V_THETA_K).to(device)
    n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
    n_total   = sum(p.numel() for p in model.parameters())
    assert _has_analytical_grad(model.V_theta)
    print(f'[SQ2 Phase-2] V_theta swapped: LowRankQuadraticVTheta(rank={V_THETA_K}), params={n_v_theta:,}')
else:
    print(f'[MLP] Keeping original MLP V_theta (Phase-2 path inactive for this run)')

HAS_STRUCTURED_VTHETA = V_THETA_KIND in ('sq1', 'sq2', 'sq3')
IS_MIXTURE = V_THETA_KIND == 'sq3'

print(f'params (after swap):  total={n_total:,}  V_theta={n_v_theta:,}  V_phi={n_v_phi:,}')
print(f'Phase-2 active: {_has_analytical_grad(model.V_theta)}')

## 5. Training loop with V_θ regularisation

In [ ]:
import math, time, json


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def tau_at(step):
    anneal_fraction = 0.8
    warm = int((1.0 - anneal_fraction) * STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    if step >= STEPS:
        return GUMBEL_TAU_MIN
    progress = (step - warm) / max(STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)


def forward_with_vreg(model, x, targets, lambda_v):
    """Decomposed forward for all model kinds."""
    h0 = model._embed(x)
    if MODEL_KIND == 'hybrid_fock_parf':
        h_attn, _ = model._attn_stack(h0)
        h_k = model.ln_boundary(h_attn)
        h_L, _ = model._stack_forward(h_k, x, return_trajectory=False)
    else:
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, model.cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


HAS_GUMBEL = hasattr(model, 'set_gumbel_tau')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

model.train()
log = []
t0 = time.time()

for step in range(STEPS):
    current_lr = lr_at(step)
    for pg in optimizer.param_groups:
        pg['lr'] = current_lr

    if HAS_GUMBEL:
        model.set_gumbel_tau(tau_at(step))

    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    optimizer.zero_grad()
    _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    if step % LOG_INTERVAL == 0:
        elapsed = time.time() - t0
        ppl_est = math.exp(min(loss_ntp.item(), 20))
        print(f'step {step:5d}/{STEPS}  loss_ntp={loss_ntp.item():.4f}  '
              f'ppl~{ppl_est:.1f}  v_reg={v_reg.item():.4f}  '
              f'lr={current_lr:.2e}  {elapsed:.0f}s')

    if step % EVAL_INTERVAL == 0 or step == STEPS - 1:
        val_loss = evaluate()
        val_ppl  = math.exp(min(val_loss, 20))
        entry = {
            'step': step, 'train_loss': loss_ntp.item(),
            'val_loss': val_loss, 'val_ppl': val_ppl,
            'v_reg': v_reg.item(), 'lr': current_lr,
            'elapsed_s': time.time() - t0,
        }
        log.append(entry)
        print(f'>>> eval step={step}  val_ppl={val_ppl:.2f}  '
              f'val_loss={val_loss:.4f}  elapsed={entry["elapsed_s"]:.0f}s')

print(f'Training complete.  Best val PPL: {min(e["val_ppl"] for e in log):.2f}')

## 6. Save checkpoint and training log

In [ ]:
full_tag = f'phase2_{CELL}_{MODEL_KIND}_d{D}_L{L}_K{V_THETA_K}'
if N_REGISTERS > 0:
    full_tag += f'_M{N_REGISTERS}'
if N_ATTN > 0:
    full_tag += f'_attn{N_ATTN}'
full_tag += f'_seed{SEED}'

ckpt_path = RESULTS_ROOT / f'{full_tag}_model.pt'
log_path  = RESULTS_ROOT / f'{full_tag}_training_log.jsonl'

torch.save(model.state_dict(), ckpt_path)
with open(log_path, 'w') as f:
    for entry in log:
        f.write(json.dumps(entry) + '\n')

print(f'Checkpoint: {ckpt_path}')
print(f'Log:        {log_path}')
print(f'Best val PPL: {min(e["val_ppl"] for e in log):.2f}')
print(f'Final val PPL: {log[-1]["val_ppl"]:.2f}')

## 7. Training curve

In [ ]:
import matplotlib.pyplot as plt

steps_arr = [e['step'] for e in log]
ppls      = [e['val_ppl'] for e in log]
ntp_losses = [e['train_loss'] for e in log]
vregs      = [e['v_reg'] for e in log]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps_arr, ppls, 'b-o', ms=3)
axes[0].axhline(7.81, color='green', ls='--', label='Attn baseline 7.81')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Val PPL (log scale)')
axes[0].set_yscale('log'); axes[0].set_title(f'{CELL}: Validation PPL')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(steps_arr, ntp_losses, 'r-', alpha=0.8)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Train NTP Loss')
axes[1].set_title(f'{CELL}: Train Loss'); axes[1].grid(True, alpha=0.3)

axes[2].plot(steps_arr, vregs, 'orange')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('V_theta reg term')
axes[2].set_title(f'{CELL}: V_θ Regularisation'); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'{recipe["desc"]}  —  best PPL={min(ppls):.2f}', fontsize=11)
plt.tight_layout()
fig_path = RESULTS_ROOT / f'{full_tag}_training_curve.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 8. V_θ landscape diagnostics (Phase-2: analytical attractors)

Because `MixtureQuadraticVTheta` has `attractor_centres(xi)`, we can read
the attractor positions directly from the model parameters — no numerical
optimisation needed.  Each of the K=8 basin centres is projected to vocabulary
space to identify the tokens each basin is "attracting" the hidden state toward.

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

v_samples = []
model.eval()

fixed_prompts = [
    'Once upon a time',
    'The little girl',
    'There was a big',
    'She smiled and',
]
prompt_ids = [
    tokenizer.encode(p, return_tensors='pt').to(device) for p in fixed_prompts
]

with torch.enable_grad():
    for _ in range(10):
        xb, _ = get_batch(val_ids, 4, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        h0 = model._embed(x)
        if MODEL_KIND == 'hybrid_fock_parf':
            h_attn, _ = model._attn_stack(h0)
            h_k = model.ln_boundary(h_attn)
            h_L, _ = model._stack_forward(h_k, x)
        else:
            h_L, _ = model._stack_forward(h0, x)
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L.detach())
        v_samples.append(V_vals.detach().cpu())

all_v = torch.cat(v_samples, dim=0).flatten()
ls_stats = {
    'mean': float(all_v.mean()),
    'std':  float(all_v.std()),
    'min':  float(all_v.min()),
    'max':  float(all_v.max()),
    'range': float(all_v.max() - all_v.min()),
}
print('V_theta landscape statistics:')
for k, v in ls_stats.items():
    print(f'  {k}: {v:.4f}')

ls_path = RESULTS_ROOT / f'{full_tag}_landscape_stats.json'
with open(ls_path, 'w') as f:
    json.dump(ls_stats, f, indent=2)

# ── Analytical attractor extraction (Phase-2 specific) ────────────────────────
attractor_info = []
if IS_MIXTURE:
    print(f'\nAnalytical attractor centres (K={V_THETA_K} basins) on fixed prompts:')
    W_emb = model.E.weight.detach().cpu()   # (V, d)

    for prompt_str, pid in zip(fixed_prompts, prompt_ids):
        with torch.enable_grad():
            h0p = model._embed(pid)
            if MODEL_KIND == 'hybrid_fock_parf':
                h_ap, _ = model._attn_stack(h0p)
                h_kp = model.ln_boundary(h_ap)
                h_Lp, _ = model._stack_forward(h_kp, pid)
            else:
                h_Lp, _ = model._stack_forward(h0p, pid)

        xi_p = causal_cumulative_mean(h_Lp.detach())
        # centres: (1, T, K, d)
        centres = model.V_theta.attractor_centres(xi_p).detach().cpu()
        T_p = centres.shape[1]
        last_pos = T_p - 1

        basin_tokens = []
        for k in range(V_THETA_K):
            c_k = centres[0, last_pos, k, :]   # (d,)
            scores = (W_emb @ c_k)             # (V,)
            top5 = scores.topk(5).indices.tolist()
            top5_str = [tokenizer.decode([t]).strip() for t in top5]
            basin_tokens.append(top5_str)

        attractor_info.append({'prompt': prompt_str, 'basins': basin_tokens})
        print(f'  prompt: {prompt_str!r}')
        for k, toks in enumerate(basin_tokens):
            print(f'    basin {k}: {toks}')

    attr_path = RESULTS_ROOT / f'{full_tag}_analytical_attractors.json'
    with open(attr_path, 'w') as f:
        json.dump(attractor_info, f, indent=2)
    print(f'Saved attractor info: {attr_path}')

## 9. Phase-2 timing benchmark

Compares per-step wall-clock time with a Phase-1-equivalent model (MLP V_θ,
legacy `autograd.grad` over the full U graph) vs. the Phase-2 SQ3 model.
Run only on a small number of steps so it completes quickly.

In [ ]:
from parf.model_parf import ScalarPotential

BENCH_STEPS = 20

def _bench_forward(m, x, y, steps=BENCH_STEPS):
    m.train()
    opt = torch.optim.SGD(m.parameters(), lr=1e-4)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(steps):
        opt.zero_grad()
        _, loss = m(x, y)
        loss.backward()
        opt.step()
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t0) / steps

torch.manual_seed(42)
xb, yb = get_batch(train_ids, BATCH, min(BLOCK, 64), rng)  # short seq for timing
x_b = torch.from_numpy(xb[:, :64]).to(device)
y_b = torch.from_numpy(yb[:, :64]).to(device)

# Phase-2 model (current, SQ3 V_theta)
t_phase2 = _bench_forward(model, x_b, y_b)

# Phase-1 equivalent: swap back to MLP V_theta temporarily
model.V_theta = ScalarPotential(D, V_HIDDEN, V_DEPTH).to(device)
t_phase1 = _bench_forward(model, x_b, y_b)

# Restore SQ3
model.V_theta = MixtureQuadraticVTheta(d=D, K=V_THETA_K).to(device)

speedup = t_phase1 / t_phase2
print(f'Phase-1 (MLP V_θ, full autograd.grad):  {t_phase1*1000:.1f} ms/step')
print(f'Phase-2 (SQ3 V_θ, analytical_grad):     {t_phase2*1000:.1f} ms/step')
print(f'Speedup: {speedup:.2f}×  (B={BATCH} T=64 d={D} L={L})')

timing = {
    'ms_per_step_phase1': t_phase1 * 1000,
    'ms_per_step_phase2': t_phase2 * 1000,
    'speedup': speedup,
    'B': BATCH, 'T': 64, 'd': D, 'L': L, 'K': V_THETA_K,
}
timing_path = RESULTS_ROOT / f'{full_tag}_timing.json'
with open(timing_path, 'w') as f:
    json.dump(timing, f, indent=2)
print(f'Saved timing: {timing_path}')

## 10. Cross-cell comparison dashboard

In [ ]:
results = {}
for cell_name in ('L1', 'L2', 'L3'):
    cell_dir = GDRIVE_OUT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    last = rows[-1] if rows else None
    best = min(r['val_ppl'] for r in rows) if rows else None
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = None
    if ls_files:
        ls = json.loads(ls_files[-1].read_text())
    timing_files = sorted(cell_dir.glob('*_timing.json'))
    timing = None
    if timing_files:
        timing = json.loads(timing_files[-1].read_text())
    results[cell_name] = {
        'desc': SCALE_RECIPES[cell_name]['desc'],
        'val_ppl': last['val_ppl'] if last else None,
        'best_ppl': best,
        'landscape': ls,
        'timing': timing,
    }

# Load v3 references (S1–S4) for comparison
V3_REF = {
    'S1': {'desc': 'v3 Reg PARF SQ3 K=4 d=256 16k', 'best_ppl': None},
    'S2': {'desc': 'v3 Reg FockPARF SQ3 K=4 d=256 16k', 'best_ppl': None},
    'S3': {'desc': 'v3 Hybrid FockPARF+Attn SQ3 K=4 d=256 16k', 'best_ppl': None},
    'S4': {'desc': 'v3 Hybrid SPLM+Attn SQ3 K=4 d=256 16k', 'best_ppl': None},
}
_v3_root = GDRIVE_OUT.parent / 'semsimula_tinystories_v3'
for _cn in V3_REF:
    _v3_dir = _v3_root / _cn / f'seed{SEED}'
    if _v3_dir.exists():
        _logs = sorted(_v3_dir.glob('*_training_log.jsonl'))
        if _logs:
            _rows = [json.loads(l) for l in _logs[-1].read_text().splitlines()]
            V3_REF[_cn]['best_ppl'] = min(r['val_ppl'] for r in _rows) if _rows else None

print(f'{"Model":<55} {"best PPL":>10} {"final PPL":>10} {"V range":>8} {"speedup":>8}')
print('-' * 100)
print(f'{"Attn baseline (8L GPT-2 d=256)":<55} {"7.81":>10} {"-":>10} {"-":>8} {"-":>8}')
print()
for _cn, _ref in V3_REF.items():
    _ppl = f'{_ref["best_ppl"]:.2f}' if _ref['best_ppl'] else 'pending'
    print(f'{"  "+_ref["desc"]:<55} {_ppl:>10}')
print()
for cell_name in ('L1', 'L2', 'L3'):
    r = results.get(cell_name)
    if r is None:
        print(f'  {cell_name}: not yet run')
        continue
    best  = f'{r["best_ppl"]:.2f}' if r['best_ppl'] else '-'
    final = f'{r["val_ppl"]:.2f}'  if r['val_ppl']  else '-'
    v_rng = f'{r["landscape"]["range"]:.3f}' if r['landscape'] else '-'
    spdup = f'{r["timing"]["speedup"]:.2f}×' if r['timing'] else '-'
    print(f'  {cell_name}: {r["desc"]:<50} {best:>10} {final:>10} {v_rng:>8} {spdup:>8}')